In [ ]:
# 78.36
# Add parent directory to path for local imports
import sys
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
# 3rd party imports
import torch
import numpy as np
from torch import nn as NN
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

# local imports
from settings import Constants
from neural_net import StockModel
from dataframe_handler import DataManager
from binance_api.indicators import ADX, RSI, EMA

In [ ]:
# --------------------------------------------------
# GLOBAL SPEED SETTINGS
# --------------------------------------------------
DEVICE = Constants.DEVICE

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

In [ ]:
# --------------------------------------------------
# HYPER PARAMETERS (Modify these as needed)
# --------------------------------------------------

# FILE paths
CSV_FILE_NAME = Constants.CSV_FILE_PATH
PROCESSED_FILE_NAME = Constants.PROCESSED_FILE_PATH

# DATASET parameters
LABEL_THRESH = Constants.LABEL_THRESH
LOOK_AHEAD_CANDLES = Constants.LOOK_AHEAD_CANDLES
TRAIN_RATIO = Constants.TRAIN_RATIO
COLS_TO_SCALE_LOG = Constants.COLS_TO_SCALE_LOG
SCALABLE_COLS = Constants.SCALABLE_COLS
COLS_TO_EXTRACT = Constants.COLS_TO_EXTRACT

# Model architecture
INPUT_SIZE = Constants.INPUT_SIZE
LSTM_HIDDEN_SIZE = Constants.LSTM_HIDDEN_SIZE
LSTM_NUM_LAYERS = Constants.LSTM_NUM_LAYERS
LSTM_DROPOUT = Constants.LSTM_DROPOUT
FC1_OUT_FEATURES = Constants.FC1_OUT_FEATURES
FC_DROPOUT = Constants.FC_DROPOUT
NUM_CLASSES = Constants.NUM_CLASSES

# Training parameters
EPOCHS = Constants.EPOCHS
BATCH_SIZE = Constants.BATCH_SIZE
LEARNING_RATE = Constants.LEARNING_RATE
GRADIENT_CLIP = Constants.GRADIENT_CLIP
VAL_PATIENCE = Constants.VAL_PATIENCE
SEQUENCE_LENGTH = Constants.SEQUENCE_LENGTH

# Loss weights for [Neutral, Up, Down]
CROSS_ENTROPY_LOSS_WEIGHTS = Constants.CROSS_ENTROPY_LOSS_WEIGHTS

# Model save path
MODEL_SAVE_PATH = Constants.MODEL_SAVE_PATH


In [ ]:
# -------------------------------------
# PROCESS RAW FILE
# -------------------------------------

# create indicators
ema_1 = EMA(20)
ema_2 = EMA(50)
rsi = RSI(14)
adx = ADX(14)

# create datafram manager
dm = DataManager(
    device=DEVICE,
    split_ratio=TRAIN_RATIO,
    sequence_length=SEQUENCE_LENGTH,
    csv_file=CSV_FILE_NAME
)

# implement preprocessing
dm.preprocess(
    indicators=[ema_1, ema_2, rsi, adx],
    threshold=LABEL_THRESH, 
    look_ahead=LOOK_AHEAD_CANDLES,
)

# implement scaling
dm.scale(
    cols=SCALABLE_COLS,
    log_cols=COLS_TO_SCALE_LOG
)

# the columns to use (direction-only, no min/max prediction)
feature_cols = ["open", "volume", "ema_0_diff", "ema_1_diff", "rsi_2", "adx_3", "return", "label"]
target_col = ["label"]
processed_tensors = dm.get_tensors(feature_cols, target_col)

# extract train and test
trainX, trainY = dm.get_train_tensors(feature_cols, target_col)
testX, testY = dm.get_test_tensors(feature_cols, target_col)

# save to a file
dm.df.to_csv(PROCESSED_FILE_NAME, index=False)
print(f"Processed data saved to {PROCESSED_FILE_NAME}")


# setup training dataset
train_dataset = TensorDataset(trainX, trainY)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

In [ ]:
# --------------------------------------------------
# TRAIN WITH EARLY STOPPING
# --------------------------------------------------

# create the model for direction classification
model = StockModel(
    input_size=INPUT_SIZE,
    lstm_layer_out=LSTM_HIDDEN_SIZE,
    lstm_layer_num=LSTM_NUM_LAYERS,
    lstm_dropout=LSTM_DROPOUT,
    fc1_out=FC1_OUT_FEATURES,
    fc_dropout=FC_DROPOUT,
    num_classes=NUM_CLASSES,
    device=DEVICE,
)

# create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# create learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=5
)

# create loss function with class weights
class_weights = torch.tensor(CROSS_ENTROPY_LOSS_WEIGHTS, dtype=torch.float, device=DEVICE)
criterion = NN.CrossEntropyLoss(weight=class_weights)

# create a scaler for mixed precision
scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))

# early stopping variables
best_val_accuracy = 0.0
patience_counter = 0
best_model_state = None

# get actual directions for validation
actual_directions = testY[:, 0, 0].cpu().tolist()

# train loop
for epoch in range(EPOCHS):

    # switch model to training
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    # get batches
    for x_batch, y_batch in train_loader:

        # reset the gradient
        optimizer.zero_grad(set_to_none=True)

        # use float16 or float32 based on the device
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):

            # model returns direction logits only
            direction_pred = model(x_batch)

            # get labels (label is now at index 0)
            labels = y_batch[:, 0, 0].long()

            # calculate loss
            loss = criterion(direction_pred, labels)

        # backward pass
        scaler.scale(loss).backward()

        # gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)

        # take a step
        scaler.step(optimizer)
        scaler.update()

        # track metrics
        total_loss += loss.item()
        preds = direction_pred.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    # calculate average loss
    avg_loss = total_loss / len(train_loader)

    # calculate train accuracy
    train_accuracy = correct / total

    # step scheduler
    scheduler.step(avg_loss)

    # === VALIDATION ===
    model.eval()
    direction_preds = []
    
    with torch.no_grad():
        for i in range(0, len(testX), BATCH_SIZE):
            x_batch = testX[i : i + BATCH_SIZE]
            direction_pred = model(x_batch)
            preds = direction_pred.argmax(dim=1).cpu().tolist()
            direction_preds.extend(preds)

    # calculate validation accuracy
    validation_accuracy = sum(
        p == a for p, a in zip(direction_preds, actual_directions)
    ) / len(actual_directions)

    # === EARLY STOPPING CHECK ===
    if validation_accuracy > best_val_accuracy:
        best_val_accuracy = validation_accuracy
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1

    # log to console
    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] Loss: {avg_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} Val Acc: {validation_accuracy:.4f}"
    )

    # if patience is greater than tolerable
    if patience_counter >= VAL_PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break

# restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model with validation accuracy: {best_val_accuracy:.4f}")

In [ ]:
# --------------------------------------------------
# FINAL EVALUATION
# --------------------------------------------------

# load model if not already loaded
if "model" not in locals():
    model = torch.jit.load(MODEL_SAVE_PATH, map_location=DEVICE)
    model.to(DEVICE)
model.eval()

direction_preds = []

# with no gradient
with torch.no_grad():

    # iterate
    for i in range(0, len(testX), BATCH_SIZE):

        # get current batch
        x_batch = testX[i : i + BATCH_SIZE]

        # model returns direction logits only
        direction_pred = model(x_batch)

        # get predicted class
        preds = direction_pred.argmax(dim=1).cpu().tolist()
        direction_preds.extend(preds)

# calculate final accuracy
actual_directions = testY[:, 0, 0].cpu().tolist()
final_accuracy = sum(p == a for p, a in zip(direction_preds, actual_directions)) / len(actual_directions)
print(f"Final Test Accuracy: {final_accuracy:.4f}")

In [ ]:
# --------------------------------------------------
# PLOT CONFUSION MATRIX
# --------------------------------------------------

# class names
class_names = ["Neutral", "Up", "Down"]

# force fixed class order
cm = confusion_matrix(actual_directions, direction_preds, labels=[0, 1, 2])

# create figure
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)

# colorbar
fig.colorbar(im, ax=ax)

# ticks & tick labels
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)

# labels
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Direction Confusion Matrix")

# write values
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

# layout
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------
# SAVE MODEL
# --------------------------------------------------

# save model
traced_model = torch.jit.script(model)
torch.jit.save(traced_model, MODEL_SAVE_PATH)
print(f"Model saved to: {MODEL_SAVE_PATH}")